In [1]:
import numpy as np
import pandas as pd

In [2]:
df=pd.read_csv("imdb_top_1000.csv")
print(df.head(10))
print(df.tail(10))
columns=["Genre" , "Director", "Star1" , "Star2" , "Star3" , "Star4" , "Series_Title"]
df_movie=pd.DataFrame(df[columns])
print(df_movie.head(10))
print(df.info())
print(f"null in df_movie \n {df_movie.isna().sum()}")
print(f"Duplicated data i9n df_movie = {df_movie.duplicated().sum()}")


                                         Poster_Link  \
0  https://m.media-amazon.com/images/M/MV5BMDFkYT...   
1  https://m.media-amazon.com/images/M/MV5BM2MyNj...   
2  https://m.media-amazon.com/images/M/MV5BMTMxNT...   
3  https://m.media-amazon.com/images/M/MV5BMWMwMG...   
4  https://m.media-amazon.com/images/M/MV5BMWU4N2...   
5  https://m.media-amazon.com/images/M/MV5BNzA5ZD...   
6  https://m.media-amazon.com/images/M/MV5BNGNhMD...   
7  https://m.media-amazon.com/images/M/MV5BNDE4OT...   
8  https://m.media-amazon.com/images/M/MV5BMjAxMz...   
9  https://m.media-amazon.com/images/M/MV5BMmEzNT...   

                                    Series_Title Released_Year Certificate  \
0                       The Shawshank Redemption          1994           A   
1                                  The Godfather          1972           A   
2                                The Dark Knight          2008          UA   
3                         The Godfather: Part II          1974         

In [3]:
for col in columns:
    if col != "Genre":
        df_movie[col] = df_movie[col].str.replace(" " , "_")
    else:
        df_movie[col] = df_movie[col].str.replace(",", " ")

df_movie["Genre"] = df_movie["Genre"].str.replace("Sci-Fi", "SciFi")
df_movie["Genre"] = df_movie["Genre"].str.replace("Film Noir", "Film_Noir")
df_movie.to_csv("df_movie.csv" ,  header=True , index = False)
print(df_movie.head(10))
print(df_movie.shape)
print(df_movie.info())


                       Genre              Director              Star1  \
0                      Drama        Frank_Darabont        Tim_Robbins   
1               Crime  Drama  Francis_Ford_Coppola      Marlon_Brando   
2       Action  Crime  Drama     Christopher_Nolan     Christian_Bale   
3               Crime  Drama  Francis_Ford_Coppola          Al_Pacino   
4               Crime  Drama          Sidney_Lumet        Henry_Fonda   
5   Action  Adventure  Drama         Peter_Jackson        Elijah_Wood   
6               Crime  Drama     Quentin_Tarantino      John_Travolta   
7  Biography  Drama  History      Steven_Spielberg        Liam_Neeson   
8   Action  Adventure  SciFi     Christopher_Nolan  Leonardo_DiCaprio   
9                      Drama         David_Fincher          Brad_Pitt   

                  Star2              Star3             Star4  \
0        Morgan_Freeman         Bob_Gunton    William_Sadler   
1             Al_Pacino         James_Caan      Diane_Keaton   
2   

In [4]:
#vectoriza
from sklearn.feature_extraction.text import CountVectorizer
weights = {
    "Genre": 3,
    "Director": 2,
    "Star1": 1.5,
    "Star2": 1,
    "Star3": 0.5,
    "Star4": 0.5, 
    "Series_Title": 0
}
vec = {}
vectorizer = {}
for col in columns:
    vectorizer[col] = CountVectorizer()
    vec[col]= vectorizer[col].fit_transform(df_movie[col])*weights[col]

for col in columns:
    print(f"{col}: {vec[col].shape}")

print(vectorizer["Genre"].get_feature_names_out())
print(type(vec[col]))

Genre: (1000, 22)
Director: (1000, 574)
Star1: (1000, 697)
Star2: (1000, 872)
Star3: (1000, 931)
Star4: (1000, 988)
Series_Title: (1000, 1131)
['action' 'adventure' 'animation' 'biography' 'comedy' 'crime' 'drama'
 'family' 'fantasy' 'film' 'history' 'horror' 'music' 'musical' 'mystery'
 'noir' 'romance' 'scifi' 'sport' 'thriller' 'war' 'western']
<class 'scipy.sparse._csr.csr_matrix'>


In [5]:
from scipy.sparse import hstack
final_vector = hstack([vec[col] for col in columns if col != "Series_Title"]) 
print(final_vector.shape)

(1000, 4084)


In [6]:
# cosine similarity
from sklearn.metrics.pairwise import  cosine_similarity
similarity = cosine_similarity(final_vector )
print(similarity)


[[1.         0.43335767 0.37304197 ... 0.37304197 0.43335767 0.        ]
 [0.43335767 1.         0.60173668 ... 0.30086834 0.34951456 0.30086834]
 [0.37304197 0.60173668 1.         ... 0.25899281 0.30086834 0.25899281]
 ...
 [0.37304197 0.30086834 0.25899281 ... 1.         0.60173668 0.        ]
 [0.43335767 0.34951456 0.30086834 ... 0.60173668 1.         0.13371926]
 [0.         0.30086834 0.25899281 ... 0.         0.13371926 1.        ]]


In [15]:
#select a movie from the list
movie_name=input("Please enter your favorite movie name")
movie_name = movie_name.replace(" ", "_")
index_name= df_movie[df_movie["Series_Title"]==movie_name].index[0]
#print(index_name)
similarity_movie = similarity[index_name]
#print(similarity_movie)
top_five=np.argsort(similarity_movie) [::-1][1:6]
#print(top_five)
top = df_movie.iloc[top_five]["Series_Title"]
#print(f"5 top recomnded movies = \n{top}")
for movie in top:
    print(movie)


Hacksaw_Ridge
The_Message
Inherit_the_Wind
La_passion_de_Jeanne_d'Arc
The_Last_King_of_Scotland


In [8]:
# Knn approach
from sklearn.neighbors import NearestNeighbors

nearest_n = NearestNeighbors(
    n_neighbors=6,
    algorithm="brute",
    metric="cosine"
)

nearest_n.fit(final_vector)

movie_name = input("Movie: ").replace(" ", "_")

index = df_movie[df_movie["Series_Title"] == movie_name].index[0]

movie_vector = final_vector[index]

distances, indices = nearest_n.kneighbors(movie_vector)

print(indices)
print(distances)

print("Recommended movies:")

for i in indices[0][1:]:
    print(df_movie.iloc[i]["Series_Title"])

[[  7 157 766 673 102 235]]
[[0.         0.22302158 0.22302158 0.22302158 0.22302158 0.22302158]]
Recommended movies:
Der_Untergang
The_Last_King_of_Scotland
Glory
Braveheart
Hotel_Rwanda
